### Cell 1: 导入所有必需的库

In [1]:
import cv2
import numpy as np
import pandas as pd
import h5py
import tarfile
import os
import datetime as dt
from math import *
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import calendar

### Cell 2: 路径配置与全局参数

In [ ]:
# ==========================================
# 🌟 干净利落的全局配置，杜绝人工挑选“典型日”
# ==========================================
project_path = os.getcwd()
tar_folder = os.path.join(project_path, '2019StanFord_image')
pv_data_path = os.path.join(project_path, 'pv_data', 'pv_output_valid.pkl') 

# 确保输出到空间充足的硬盘
output_folder = r"E:\造数据集\output_folder_v3"
os.makedirs(output_folder, exist_ok=True)

# 升级为 V3，防止与你之前的旧文件冲突
output_h5_path = os.path.join(output_folder, '2019_dataset_dual_V3.h5')

# ==========================================
# 🌟 恢复你的晴天标定日 (纯用于硬件校准)
# ==========================================
sunny_calibration_days = [
    (2019,1,25), (2019,5,31), (2019,6,23), 
    (2019,7,14), (2019,8,11), (2019,9,6), (2019,10,14)
]

print(f"✅ 配置文件加载完成！HDF5 将输出至: {output_h5_path}")

✅ 配置文件加载完成！HDF5 将输出至: E:\造数据集\output_folder_v3\2019_dataset_dual_V3.h5


### Cell 3: 太阳位置计算与插值台账提取

In [ ]:
# ==========================================
# 🌟 Cell 3: 物理约束引擎 (包含镜头畸变标定与插值)
# ==========================================

# ... (doy_tod_conv, solar_angle, get_theo_sun_position 函数保持完全不变，放在这里) ...
def doy_tod_conv(date_and_time, longitude=-122.174199, time_zone_center_longitude=-120):
    pst_center_longitude = time_zone_center_longitude 
    loc_longitude = longitude 
    correction = np.abs(60/15*(loc_longitude - pst_center_longitude))
    min_correction = int(correction) 
    sec_correction = int((correction - min_correction)*60)  
    if date_and_time.minute <= min_correction:
        date_and_time = date_and_time.replace(hour=date_and_time.hour-1, minute=60+date_and_time.minute-min_correction-1, second=60-sec_correction)
    else:
        date_and_time = date_and_time.replace(minute=date_and_time.minute-min_correction-1, second=60-sec_correction)
    time_of_day = date_and_time.hour * 3600 + date_and_time.minute * 60 + date_and_time.second
    months = [31,28,31,30,31,30,31,31,30,31,30,31] 
    if (date_and_time.year % 4 == 0) and (date_and_time.year % 100 != 0 or date_and_time.year % 400 == 0): months[1] = 29 
    day_of_year = sum(months[:date_and_time.month-1]) + date_and_time.day
    return day_of_year, time_of_day

def solar_angle(times, latitude=37.424107, longitude=-122.174199, time_zone_center_longitude=-120):
    day_of_year, time_of_day = doy_tod_conv(times, longitude, time_zone_center_longitude)
    latitude = radians(latitude) 
    B = 2 * pi * (day_of_year - 80) / 365.0
    eot_minutes = 9.87 * sin(2 * B) - 7.53 * cos(B) - 1.5 * sin(B)
    time_of_day_corrected = time_of_day + (eot_minutes * 60)
    alpha = 2*pi*(time_of_day_corrected-43200)/86400 
    delta = radians(23.44*sin(radians((360/365.25)*(day_of_year-80))))
    chi = acos(sin(delta)*sin(latitude)+cos(delta)*cos(latitude)*cos(alpha))
    tan_xi = sin(alpha)/(sin(latitude)*cos(alpha)-cos(latitude)*tan(delta)) 
    if alpha>0 and tan_xi>0: xi = pi+atan(tan_xi)
    elif alpha>0 and tan_xi<0: xi = 2*pi+atan(tan_xi)
    elif alpha<0 and tan_xi>0: xi = atan(tan_xi)
    else: xi = pi+atan(tan_xi)
    return degrees(xi), degrees(chi)

def get_theo_sun_position(time_obj):
    delta, r, origin_y, origin_x = 14.036, 928, 928, 960  
    azimuth, zenith = solar_angle(time_obj) 
    rho = zenith / 90 * r 
    theta = azimuth - delta + 90 
    theo_y = origin_y - rho * sin(radians(theta))
    theo_x = origin_x + rho * cos(radians(theta))
    return int(round(theo_x)), int(round(theo_y))



MONTHLY_INTERP_TABLE = {}
print("⏳ 正在使用极其纯净的晴天基准日，进行镜头畸变物理标定...")

for year, month, day in sunny_calibration_days:
    month_int = month
    date_str = f"{year}{month:02d}{day:02d}"
    tar_name = f"2019_{month:02d}_images_raw.tar"
    tar_path = os.path.join(tar_folder, tar_name)
    
    if not os.path.exists(tar_path): 
        print(f"⚠️ 跳过 {date_str}，找不到对应的 tar 包")
        continue
        
    time_floats, dx_list, dy_list = [], [], []
    with tarfile.open(tar_path, "r") as tar:
        day_members = [m for m in tar.getmembers() if m.name.endswith('.jpg') and date_str in m.name]
        day_members.sort(key=lambda x: os.path.basename(x.name))
        
        # 每隔 15 张图采样一次，计算当前月的镜头畸变偏移量 (dx, dy)
        for m in day_members[::15]:
            curr_time = dt.datetime.strptime(os.path.basename(m.name).split('.')[0], '%Y%m%d%H%M%S')
            if curr_time.hour < 7 or curr_time.hour > 18: continue
                
            img_raw = cv2.imdecode(np.asarray(bytearray(tar.extractfile(m).read()), dtype=np.uint8), cv2.IMREAD_COLOR)
            gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
            theo_x, theo_y = get_theo_sun_position(curr_time)
            
            search_radius = 120
            if theo_y - search_radius < 0 or theo_y + search_radius > 2048 or theo_x - search_radius < 0 or theo_x + search_radius > 2048:
                continue
                
            roi = gray[theo_y - search_radius : theo_y + search_radius, theo_x - search_radius : theo_x + search_radius]
            _, thresh = cv2.threshold(roi, 245, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(cv2.erode(thresh, np.ones((7, 7), np.uint8), iterations=2), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            if contours:
                largest_contour = max(contours, key=cv2.contourArea)
                if cv2.contourArea(largest_contour) > 50:
                    (cX_local, cY_local), _ = cv2.minEnclosingCircle(largest_contour)
                    time_float = curr_time.hour + curr_time.minute / 60.0 + curr_time.second / 3600.0
                    time_floats.append(round(time_float, 3))
                    
                    # 记录理论值与 CV 提取出的真实值之间的偏移
                    dx_list.append(round((theo_x - search_radius + cX_local) - theo_x, 1))
                    dy_list.append(round((theo_y - search_radius + cY_local) - theo_y, 1))
    
    if time_floats:
        MONTHLY_INTERP_TABLE[month_int] = {'times': time_floats, 'dx': dx_list, 'dy': dy_list}

# ==========================================
# 🌟 缺失月份的稳妥回退机制 (Fallback)
# ==========================================
# 比如 2月、3月、4月 没有晴天标定数据，我们用离它最近的 1 月或 5 月的镜头畸变数据
FALLBACK_MONTHS = { 2: 1, 3: 1, 4: 1, 11: 10, 12: 10 }

def get_smart_sun_position(time_obj):
    theo_x, theo_y = get_theo_sun_position(time_obj)
    month = time_obj.month
    
    # 查找标定台账：如果有当月的用当月，没有当月用回退字典，兜底用1月
    ref_month = month if month in MONTHLY_INTERP_TABLE else FALLBACK_MONTHS.get(month, 1)
    
    # 防止字典里完全没数据报错
    if ref_month not in MONTHLY_INTERP_TABLE:
        return theo_x, theo_y
        
    table = MONTHLY_INTERP_TABLE[ref_month]
    t_curr = time_obj.hour + time_obj.minute / 60.0 + time_obj.second / 3600.0
    
    # 插值计算偏移量
    dx = np.interp(t_curr, table['times'], table['dx'])
    dy = np.interp(t_curr, table['times'], table['dy'])
    
    return int(round(theo_x + dx)), int(round(theo_y + dy))

print("✅ 镜头畸变标定完成！CV 坐标提取的鲁棒性已拉满。")

⏳ 正在使用极其纯净的晴天基准日，进行镜头畸变物理标定...
✅ 镜头畸变标定完成！CV 坐标提取的鲁棒性已拉满。


### Cell 4: 图像双分支采样核心

In [4]:
# ==========================================
# 🌟 高效视场重采样引擎 (严格遵循 PyTorch C,H,W 格式)
# ==========================================
def extract_dual_branch_images(img_bgr, sun_x, sun_y):
    mask = np.zeros(img_bgr.shape[:2], dtype=np.uint8)
    cv2.circle(mask, (1053, 1024), 992, 255, -1)
    img_clean = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)
    
    # 1. 全局图 (缩放并转换通道)
    global_img_rgb = cv2.cvtColor(cv2.resize(img_clean, (256, 256), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    img_g_pt = global_img_rgb.transpose(2, 0, 1) # -> (3, 256, 256)
    
    # 2. 局部图 (极宽容裁剪法)
    HUGE_CROP = 768
    HALF_CROP = HUGE_CROP // 2
    canvas = np.zeros((HUGE_CROP, HUGE_CROP, 3), dtype=np.uint8)
    
    y_min, y_max = sun_y - HALF_CROP, sun_y + HALF_CROP
    x_min, x_max = sun_x - HALF_CROP, sun_x + HALF_CROP
    img_h, img_w = img_clean.shape[:2]
    
    img_y_min, img_y_max = max(0, y_min), min(img_h, y_max)
    img_x_min, img_x_max = max(0, x_min), min(img_w, x_max)
    
    if img_y_min < img_y_max and img_x_min < img_x_max:
        canvas_y_min, canvas_x_min = img_y_min - y_min, img_x_min - x_min
        canvas[canvas_y_min:canvas_y_min+(img_y_max-img_y_min), canvas_x_min:canvas_x_min+(img_x_max-img_x_min)] = img_clean[img_y_min:img_y_max, img_x_min:img_x_max]
        
    local_img_rgb = cv2.cvtColor(cv2.resize(canvas, (128, 128), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    img_l_pt = local_img_rgb.transpose(2, 0, 1) # -> (3, 128, 128)
    
    return img_g_pt, img_l_pt

### Cell 5: 终极 HDF5 写入与时间轴切分引擎

In [5]:
# ==========================================
# 🚀 极致性能的单表直写引擎 + 自动时序切分
# ==========================================
pv_output_all = pd.read_pickle(pv_data_path)

def find_time_within_pdseries(time_array, time_point):
    probable_idx = np.searchsorted(time_array, time_point)
    if probable_idx < len(time_array) and time_array[probable_idx] == time_point: return probable_idx
    return None

# 🌟 内存缓冲机制：凑够 100 帧再写入硬盘，速度提升 100 倍且绝不产生碎片！
BUFFER_SIZE = 100
buffers = {'global': [], 'local': [], 'pv': [], 'sun': []}
all_recorded_times = []

def flush_buffer(grp):
    if not buffers['pv']: return
    n_new = len(buffers['pv'])
    curr_len = grp['pv_log'].shape[0]
    
    for key in ['global_images_log', 'local_images_log', 'pv_log', 'sun_pos']:
        grp[key].resize(curr_len + n_new, axis=0)
        
    grp['global_images_log'][curr_len:] = np.stack(buffers['global'])
    grp['local_images_log'][curr_len:] = np.stack(buffers['local'])
    grp['pv_log'][curr_len:] = np.array(buffers['pv'], dtype=np.float32)
    grp['sun_pos'][curr_len:] = np.array(buffers['sun'], dtype=np.int32)
    
    for k in buffers.keys(): buffers[k].clear()

with h5py.File(output_h5_path, 'w') as h5f:
    # 🌟 彻底抛弃 test 组，所有数据汇聚于一个最高效的 trainval 大池子中
    grp = h5f.create_group('trainval')
    grp.create_dataset('global_images_log', shape=(0, 3, 256, 256), maxshape=(None, 3, 256, 256), dtype='uint8', chunks=(BUFFER_SIZE, 3, 256, 256), compression='lzf')
    grp.create_dataset('local_images_log', shape=(0, 3, 128, 128), maxshape=(None, 3, 128, 128), dtype='uint8', chunks=(BUFFER_SIZE, 3, 128, 128), compression='lzf')
    grp.create_dataset('pv_log', shape=(0,), maxshape=(None,), dtype='float32', chunks=(BUFFER_SIZE,))
    grp.create_dataset('sun_pos', shape=(0, 2), maxshape=(None, 2), dtype='int32', chunks=(BUFFER_SIZE, 2))

    tar_files = sorted([f for f in os.listdir(tar_folder) if f.endswith('.tar')])
    
    for tar_name in tar_files:
        print(f"\n--- ⚡ 正在极速解析: {tar_name} ---")
        with tarfile.open(os.path.join(tar_folder, tar_name), "r") as tar:
            members = sorted([m for m in tar.getmembers() if m.name.endswith('.jpg')], key=lambda x: x.name)
            prev_img_gray, current_day = None, None
            
            for m in tqdm(members, desc="提取与写入"):
                try:
                    curr_time = dt.datetime.strptime(os.path.basename(m.name).split('.')[0], '%Y%m%d%H%M%S')
                except: continue
                
                if curr_time.date() != current_day:
                    current_day, prev_img_gray = curr_time.date(), None
                
                # 光照过滤
                if curr_time.hour < 7 or (curr_time.hour == 7 and curr_time.minute < 30) or \
                   curr_time.hour > 17 or (curr_time.hour == 17 and curr_time.minute > 30): continue
                
                pv_idx = find_time_within_pdseries(pv_output_all.index, curr_time)
                if pv_idx is None: continue 
                
                # 内存解压
                img_raw = cv2.imdecode(np.asarray(bytearray(tar.extractfile(m).read()), dtype=np.uint8), cv2.IMREAD_COLOR)
                
                # 卡死帧过滤
                curr_img_gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
                if prev_img_gray is not None and np.sum(np.abs(curr_img_gray.astype(np.int16) - prev_img_gray.astype(np.int16))) == 0: continue
                prev_img_gray = curr_img_gray
                
                # 计算特征与坐标 (🌟 无需事后打补丁，直接算好存入)
                sun_x, sun_y = get_smart_sun_position(curr_time)
                img_g_pt, img_l_pt = extract_dual_branch_images(img_raw, sun_x, sun_y)
                
                # 填入缓冲区
                buffers['global'].append(img_g_pt)
                buffers['local'].append(img_l_pt)
                buffers['pv'].append(float(pv_output_all.iloc[pv_idx]))
                buffers['sun'].append([sun_x, sun_y])
                all_recorded_times.append(curr_time)
                
                if len(buffers['pv']) >= BUFFER_SIZE: flush_buffer(grp)
                    
    flush_buffer(grp) # 刷入最后残留的数据

# ==========================================
# 🌟 自动生成严格时间轴的物理索引
# ==========================================
print("\n🔄 正在按照 8:2 比例严谨切分时间轴...")
split_idx = int(len(all_recorded_times) * 0.8)

np.save(os.path.join(output_folder, 'times_trainval.npy'), np.array(all_recorded_times[:split_idx]))
np.save(os.path.join(output_folder, 'times_test.npy'), np.array(all_recorded_times[split_idx:]))

print("✅ HDF5 数据集构建完毕！")
print(f"📊 数据库行数: {len(all_recorded_times)} 行连续数据")
print(f"📅 训练/验证集分配了前 {split_idx} 帧，测试集分配了剩余的 {len(all_recorded_times)-split_idx} 帧。")
print("🎉 大功告成！文件自带无碎片化体质，可直接送入 PGMM-T 开启极限训练！")


--- ⚡ 正在极速解析: 2019_01_images_raw.tar ---


提取与写入:   0%|          | 0/3935 [00:00<?, ?it/s]


--- ⚡ 正在极速解析: 2019_02_images_raw.tar ---


提取与写入:   0%|          | 0/4733 [00:00<?, ?it/s]


--- ⚡ 正在极速解析: 2019_05_images_raw.tar ---


...


--- ⚡ 正在极速解析: 2019_09_images_raw.tar ---


提取与写入:   0%|          | 0/25302 [00:00<?, ?it/s]


--- ⚡ 正在极速解析: 2019_10_images_raw.tar ---


提取与写入:   0%|          | 0/26084 [00:00<?, ?it/s]


🔄 正在按照 8:2 比例严谨切分时间轴...
✅ HDF5 数据集构建完毕！
📊 数据库行数: 101368 行连续数据
📅 训练/验证集分配了前 81094 帧，测试集分配了剩余的 20274 帧。
🎉 大功告成！文件自带无碎片化体质，可直接送入 PGMM-T 开启极限训练！
